# Full Pipeline — Medical Reasoning Fine-Tuning (Track A + Track B)

This notebook combines Notebooks 02–06 into a single end-to-end run.

**Sections**
1. Bootstrap (clone, install, secrets)
2. Configuration summary
3. Train Track B (answer only)
4. Train Track A (short clinical CoT)
5. Load shared test split
6. Inference (both adapters)
7. Automatic metrics (EM + ROUGE-L)
8. LLM judge
9. Safety audit templates
10. Final report
11. Save outputs (zip for download)

## Required Kaggle environment
- Accelerator: **GPU T4 x1** or **GPU T4 x2** (dual-GPU auto-detected; uses `accelerate launch` for 2x training speed)
- Internet: **On**
- Secrets: `HF_TOKEN` (read+write), `WANDB_API_KEY`, at least one of `CEREBRAS_API_KEY` / `GROQ_API_KEY` / `GEMINI_API_KEY`

## 1. Bootstrap

In [ ]:
import warnings, os, sys, subprocess, logging
from pathlib import Path

# ── Python warnings (same-process) ────────────────────────────────────────
_W = [
    '.*AttentionMaskConverter.*', '.*use_return_dict.*', '.*max_new_tokens.*',
    '.*max_length.*max_new_tokens.*', '.*has new PAD/BOS/EOS tokens.*',
    '.*Will smartly offload gradients.*', '.*use_cache.*gradient.*checkpointing.*',
    '.*gradient_checkpointing_enable.*', '.*Trainer.tokenizer.*deprecated.*',
    '.*processing_class.*', '.*copying from a non-meta.*', '.*TypedStorage is deprecated.*',
]
for _pat in _W:
    warnings.filterwarnings('ignore', message=_pat)
warnings.filterwarnings('ignore', category=FutureWarning,      module='transformers')
warnings.filterwarnings('ignore', category=FutureWarning,      module='peft')
warnings.filterwarnings('ignore', category=FutureWarning,      module='trl')
warnings.filterwarnings('ignore', category=FutureWarning,      module='torch')
warnings.filterwarnings('ignore', category=UserWarning,        module='torch')
warnings.filterwarnings('ignore', category=DeprecationWarning, module='pkg_resources')

# ── Propagate suppression to all subprocesses (train_sft.py, llm_judge.py) ─
os.environ['PYTHONWARNINGS']        = 'ignore'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# ── HuggingFace logging-based warnings (use logging, not warnings.warn) ────
for _lg in ('transformers', 'datasets', 'peft', 'accelerate', 'huggingface_hub'):
    logging.getLogger(_lg).setLevel(logging.ERROR)

# ── Repo bootstrap ─────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/abhishek1998s/medical-reasoning-llm.git'
REPO_DIR = '/kaggle/working/medical-reasoning-llm'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [2]:
!rm -rf /kaggle/working/unsloth_compiled_cache
!pip install -q --upgrade \
    unsloth==2026.4.8 \
    transformers==5.5.0 \
    trl==0.24.0 \
    peft==0.19.1 \
    bitsandbytes==0.49.2 \
    accelerate==1.13.0 \
    datasets==4.3.0 \
    wandb==0.19.4 \
    pyyaml \
    openai \
    google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 28.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 101.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 76.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.3/806.3 kB 46.2 MB/s eta 0:00:00

In [3]:
import importlib

versions = {}
for pkg in ['unsloth', 'transformers', 'trl', 'peft',
            'datasets', 'bitsandbytes', 'accelerate', 'wandb']:
    try:
        m = importlib.import_module(pkg)
        versions[pkg] = getattr(m, '__version__', '?')
    except Exception as e:
        versions[pkg] = f'FAILED: {e}'

for k, v in versions.items():
    print(f'  {k:20s} {v}')

failed = [k for k, v in versions.items() if str(v).startswith('FAILED')]
if failed:
    raise RuntimeError(f'Failed imports: {failed}. Restart kernel and re-run.')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
  unsloth              2026.4.8
  transformers         5.5.0
  trl                  0.24.0
  peft                 0.19.1
  datasets             4.3.0
  bitsandbytes         0.49.2
  accelerate           1.13.0
  wandb                0.19.4


In [4]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login as hf_login

secrets = UserSecretsClient()

def _try_get(name):
    try:
        return secrets.get_secret(name)
    except Exception as e:
        print(f'  [skip] {name}: {e.__class__.__name__}')
        return None

os.environ['HF_TOKEN']          = _try_get('HF_TOKEN')          or ''
os.environ['WANDB_API_KEY']     = _try_get('WANDB_API_KEY')     or ''
os.environ['CEREBRAS_API_KEY']  = _try_get('CEREBRAS_API_KEY')  or ''
os.environ['GROQ_API_KEY']      = _try_get('GROQ_API_KEY')      or ''
os.environ['GEMINI_API_KEY']    = _try_get('GEMINI_API_KEY')    or ''

print('HF_TOKEN set:         ', bool(os.environ['HF_TOKEN']))
print('WANDB_API_KEY set:    ', bool(os.environ['WANDB_API_KEY']))
print('CEREBRAS_API_KEY set: ', bool(os.environ['CEREBRAS_API_KEY']))
print('GROQ_API_KEY set:     ', bool(os.environ['GROQ_API_KEY']))
print('GEMINI_API_KEY set:   ', bool(os.environ['GEMINI_API_KEY']))

if os.environ['HF_TOKEN']:
    hf_login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF login OK')

if not any([os.environ['CEREBRAS_API_KEY'],
            os.environ['GROQ_API_KEY'],
            os.environ['GEMINI_API_KEY']]):
    print('\nWARNING: no judge API key — Section 8 (LLM Judge) will be skipped.')

HF_TOKEN set:          True
WANDB_API_KEY set:     True
CEREBRAS_API_KEY set:  True
GROQ_API_KEY set:      True
GEMINI_API_KEY set:    True
HF login OK


## 2. Configuration

In [5]:
import yaml, json
import pandas as pd

cfg = yaml.safe_load(open('configs/experiment_config.yaml', encoding='utf-8'))

hub_repo_a = f"{cfg['hub']['username']}/{cfg['hub']['repos']['trackA']}"
hub_repo_b = f"{cfg['hub']['username']}/{cfg['hub']['repos']['trackB']}"

print('=== Experiment Config Summary ===')
print(f"  model:           {cfg['model']['name']}")
print(f"  max_seq_length:  {cfg['model']['max_seq_length']}")
print(f"  num_train:       {cfg['dataset']['num_train']}")
print(f"  num_val:         {cfg['dataset']['num_val']}")
print(f"  num_test:        {cfg['dataset']['num_test']}")
print(f"  max_rows:        {cfg['dataset'].get('max_rows')}")
print(f"  batch_size:      {cfg['training']['per_device_train_batch_size']}")
print(f"  grad_accum:      {cfg['training']['gradient_accumulation_steps']}")
print(f"  hub_repo_a:      {hub_repo_a}")
print(f"  hub_repo_b:      {hub_repo_b}")

=== Experiment Config Summary ===
  model:           unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit
  max_seq_length:  2048
  num_train:       100
  num_val:         10
  num_test:        5
  max_rows:        10000
  batch_size:      2
  grad_accum:      2
  hub_repo_a:      kabhisheks/qwen25-1.5b-medreason-trackA-dryrun
  hub_repo_b:      kabhisheks/qwen25-1.5b-medreason-trackB-dryrun


In [ ]:
# ── Phase 1 Deliverable: Design Document ──────────────────────────────────
_dd = Path('design_doc.md')
if _dd.exists():
    lines = _dd.read_text(encoding='utf-8').splitlines()
    # Print headers and first line of each section
    print('design_doc.md — section overview:\n')
    for ln in lines:
        if ln.startswith('#'):
            print(ln)
    print(f'\nFull document: {_dd.resolve()}  ({len(lines)} lines)')
else:
    print('WARNING: design_doc.md not found — Phase 1 deliverable missing!')

## 3. Train Track B — Answer Only

Track B is the **baseline ablation**: the model learns to output only the final answer,
with no reasoning. This gives us a clean comparison point for Track A.

In [ ]:
import torch as _torch

_n_gpus = _torch.cuda.device_count()
print(f'Detected {_n_gpus} GPU(s)')

# Use accelerate launch for multi-GPU DDP (2x speed on T4 x2); fall back to python on single GPU.
if _n_gpus > 1:
    _launcher = f'accelerate launch --num_processes {_n_gpus} --mixed_precision bf16 '
    print(f'Multi-GPU mode: {_launcher.strip()}')
else:
    _launcher = 'python '

_max_rows_flag = (
    f" --max_rows {cfg['dataset']['max_rows']}"
    if cfg['dataset'].get('max_rows') else ''
)
cmd_b = (
    _launcher
    + f"train_sft.py"
    f" --track B"
    f" --num_samples {cfg['dataset']['num_train']}"
    f" --max_seq_length {cfg['model']['max_seq_length']}"
    f" --batch_size {cfg['training']['per_device_train_batch_size']}"
    f" --grad_accum {cfg['training']['gradient_accumulation_steps']}"
    f" --epochs {cfg['training']['epochs']}"
    f" --output_dir outputs/trackB"
    f" --run_name {cfg['logging']['wandb_runs']['trackB']}"
    f" --push_to_hub"
    f" --hub_repo {hub_repo_b}"
    + _max_rows_flag
)
print('Running:', cmd_b)
ret = os.system(cmd_b)
if ret != 0:
    raise RuntimeError(f'train_sft.py (Track B) exited with code {ret}')

In [7]:
adapter_b = Path('outputs/trackB/final_adapter')
assert adapter_b.exists(), 'Track B adapter missing — check training output'
meta_b = json.loads((adapter_b / 'training_meta.json').read_text())
print(json.dumps(meta_b, indent=2))
assert meta_b.get('track') == 'B'
assert meta_b.get('train_loss', 99) < 5.0, f"train_loss high: {meta_b.get('train_loss')}"
print(f"\nTrack B train_loss = {meta_b['train_loss']:.4f}  ✓")

{
  "model": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
  "track": "B",
  "num_samples": 100,
  "max_seq_length": 2048,
  "batch_size": 2,
  "grad_accum": 2,
  "lr": 0.0002,
  "epochs": 1.0,
  "lora_r": 16,
  "lora_alpha": 16,
  "chat_template": "qwen-2.5",
  "train_runtime_sec": 158.8829,
  "train_loss": 1.6065631675720216,
  "eval_loss": 1.5081418752670288,
  "dataset_stats": {
    "n_train": 100,
    "n_eval": 5,
    "max_rows_cap": 10000,
    "cot_budget_tokens": null
  },
  "packages": {
    "unsloth": "2026.4.8",
    "transformers": "5.5.0",
    "trl": "0.24.0",
    "peft": "0.19.1"
  }
}

Track B train_loss = 1.6066  ✓


## 4. Train Track A — Short Clinical CoT

Track A is the **primary system**: the model outputs a short clinical rationale (≤150 tokens,
sentence-boundary truncated) followed by the final answer.

```
Clinical rationale:
<truncated reasoning>

Final answer:
<answer>
```

In [ ]:
cmd_a = (
    _launcher
    + f"train_sft.py"
    f" --track A_short"
    f" --num_samples {cfg['dataset']['num_train']}"
    f" --short_cot_tokens {cfg['dataset']['short_cot_max_tokens']}"
    f" --max_seq_length {cfg['model']['max_seq_length']}"
    f" --batch_size {cfg['training']['per_device_train_batch_size']}"
    f" --grad_accum {cfg['training']['gradient_accumulation_steps']}"
    f" --epochs {cfg['training']['epochs']}"
    f" --output_dir outputs/trackA"
    f" --run_name {cfg['logging']['wandb_runs']['trackA']}"
    f" --push_to_hub"
    f" --hub_repo {hub_repo_a}"
    + _max_rows_flag
)
print('Running:', cmd_a)
ret = os.system(cmd_a)
if ret != 0:
    raise RuntimeError(f'train_sft.py (Track A) exited with code {ret}')

In [9]:
adapter_a = Path('outputs/trackA/final_adapter')
assert adapter_a.exists(), 'Track A adapter missing — check training output'
meta_a = json.loads((adapter_a / 'training_meta.json').read_text())
print(json.dumps(meta_a, indent=2))
assert meta_a.get('track') == 'A_short'
assert meta_a.get('train_loss', 99) < 5.0, f"train_loss high: {meta_a.get('train_loss')}"
print(f"\nTrack A train_loss = {meta_a['train_loss']:.4f}  ✓")

{
  "model": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
  "track": "A_short",
  "num_samples": 100,
  "max_seq_length": 2048,
  "batch_size": 2,
  "grad_accum": 2,
  "lr": 0.0002,
  "epochs": 1.0,
  "lora_r": 16,
  "lora_alpha": 16,
  "chat_template": "qwen-2.5",
  "train_runtime_sec": 159.0142,
  "train_loss": 1.6470238304138183,
  "eval_loss": 1.5499441623687744,
  "dataset_stats": {
    "n_train": 100,
    "n_eval": 5,
    "max_rows_cap": 10000,
    "cot_budget_tokens": 150
  },
  "packages": {
    "unsloth": "2026.4.8",
    "transformers": "5.5.0",
    "trl": "0.24.0",
    "peft": "0.19.1"
  }
}

Track A train_loss = 1.6470  ✓


## 5. Load Shared Test Split

Both tracks are evaluated on **exactly the same test rows** — same shuffle seed, same indices.
This is what makes the A/B comparison valid.

In [11]:
from datasets import load_dataset
from transformers import AutoTokenizer
from src.splits import shuffle_filter_split

hf_token = os.environ.get('HF_TOKEN') or None

tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct', token=hf_token)
ds  = load_dataset(cfg['dataset']['name'], split=cfg['dataset']['split'],
                   token=hf_token)

_, _, test_ds = shuffle_filter_split(
    ds,
    shuffle_seed=cfg['dataset']['shuffle_seed'],
    num_train=cfg['dataset']['num_train'],
    num_val=cfg['dataset']['num_val'],
    num_test=cfg['dataset']['num_test'],
    tokenizer=tok,
    max_total_tokens=3500,
    max_rows=cfg['dataset'].get('max_rows'),
)
print(f'Test rows: {len(test_ds)}')

# Save frozen split indices so report/judge cells can reload without re-running split
_split_path = Path('outputs/test_split_indices.json')
_split_path.parent.mkdir(parents=True, exist_ok=True)
_split_path.write_text(json.dumps({
    'indices': (
        test_ds._indices.to_pylist()
        if hasattr(test_ds, '_indices') and test_ds._indices is not None
        else list(range(len(test_ds)))
    ),
    'shuffle_seed': cfg['dataset']['shuffle_seed'],
    'num_test': cfg['dataset']['num_test'],
    'max_rows': cfg['dataset'].get('max_rows'),
}, indent=2))
print(f'Split indices saved -> {_split_path}')

Test rows: 5
Split indices saved -> outputs/test_split_indices.json


## 6. Inference

Load each adapter in turn, generate predictions on the test set, and save a CSV per track.
Each row records the question, gold reference, model prediction, token counts, and latency.

In [ ]:
import gc
import torch
import transformers
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from peft import PeftModel
from src.inference import build_prediction_row, generate_with_logging

# Silence transformers' generate() logging warnings (max_new_tokens vs max_length etc.)
transformers.logging.set_verbosity_error()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

def load_adapter(adapter_id):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=cfg['model']['name'],
        max_seq_length=cfg['model']['max_seq_length'],
        dtype=None,
        load_in_4bit=True,
    )
    tokenizer = get_chat_template(tokenizer, chat_template=cfg['model']['chat_template'])
    model = PeftModel.from_pretrained(model, adapter_id, token=hf_token)
    FastLanguageModel.for_inference(model)
    return model, tokenizer

def run_track(track_name, adapter_id, out_csv):
    print(f'\n[Track {track_name}] loading: {adapter_id}')
    model, tokenizer = load_adapter(adapter_id)
    cfg_key = 'track_A' if track_name == 'A' else 'track_B'
    max_new = cfg['inference']['max_new_tokens'][cfg_key]
    rows = []
    for i, row in enumerate(test_ds):
        user = next(m for m in row['messages'] if m['role'] == 'user')
        asst = next(m for m in row['messages'] if m['role'] == 'assistant')
        gen = generate_with_logging(
            model, tokenizer, user['content'],
            max_new_tokens=max_new,
            temperature=cfg['inference']['temperature'],
            do_sample=cfg['inference']['do_sample'],
            repetition_penalty=cfg['inference']['repetition_penalty'],
            device=device,
        )
        rows.append(build_prediction_row(
            sample_id=i,
            question=user['content'],
            reference=(asst.get('content') or '').strip(),
            track_name=track_name,
            model_id=cfg['model']['name'],
            adapter_id=adapter_id,
            generation=gen,
        ))
        if (i + 1) % 10 == 0:
            print(f'  {i+1}/{len(test_ds)} done')
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    result = pd.DataFrame(rows)
    result.to_csv(out_csv, index=False)
    print(f'  saved -> {out_csv}')
    # Free GPU memory before loading the next adapter
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

pred_a = run_track('A', hub_repo_a, 'outputs/trackA/predictions.csv')
pred_b = run_track('B', hub_repo_b, 'outputs/trackB/predictions.csv')

print(f'\npred_a: {len(pred_a)} rows')
print(f'pred_b: {len(pred_b)} rows')
pred_a.head(2)

## 7. Automatic Metrics

Exact Match and ROUGE-L are computed on the **extracted final answer only** — not the
reasoning prefix — so the comparison is fair across Track A and Track B.

In [13]:
from src.data_formatting import extract_answer_for_scoring
from src.metrics import compute_core_metrics

def score_file(path, track):
    df = pd.read_csv(path)
    preds = [extract_answer_for_scoring(p, track) for p in df['prediction']]
    refs  = list(df['reference'])
    core  = compute_core_metrics(preds, refs)
    core['mean_output_tokens']     = round(float(df['output_tokens'].mean()), 2)
    core['mean_generation_time_s'] = round(float(df['generation_time_s'].mean()), 3)
    core['mean_tokens_per_sec']    = round(float(df['tokens_per_sec'].mean()), 2)
    return core

metrics_summary = {
    'trackA': score_file('outputs/trackA/predictions.csv', 'A'),
    'trackB': score_file('outputs/trackB/predictions.csv', 'B'),
}
Path('outputs/metrics_summary.json').write_text(
    json.dumps(metrics_summary, indent=2), encoding='utf-8'
)

print('=== Metrics Summary ===')
print(json.dumps(metrics_summary, indent=2))
pd.DataFrame(metrics_summary).T

=== Metrics Summary ===
{
  "trackA": {
    "exact_match": 0,
    "rouge_l": 0.08163897195901598,
    "n": 5,
    "mean_output_tokens": 216.2,
    "mean_generation_time_s": 12.103,
    "mean_tokens_per_sec": 17.6
  },
  "trackB": {
    "exact_match": 0,
    "rouge_l": 0.08447283839127975,
    "n": 5,
    "mean_output_tokens": 160.0,
    "mean_generation_time_s": 8.736,
    "mean_tokens_per_sec": 18.0
  }
}


,exact_match,rouge_l,n,mean_output_tokens,mean_generation_time_s,mean_tokens_per_sec
trackA,0.0,0.081639,5.0,216.2,12.103,17.6
trackB,0.0,0.084473,5.0,160.0,8.736,18.0


In [14]:
# Inference diagnostics: truncation rate, finish-reason distribution, length percentiles
from src.metrics import compute_operational_stats

for track_name, csv_path in [('A', 'outputs/trackA/predictions.csv'),
                               ('B', 'outputs/trackB/predictions.csv')]:
    df = pd.read_csv(csv_path)
    stats = compute_operational_stats(df)
    print(f'\n=== Track {track_name} Inference Diagnostics ===')
    print(f"  truncation_rate:           {stats.get('truncation_rate', 'N/A'):.1%}")
    print(f"  empty_prediction_rate:     {stats.get('empty_prediction_rate', 'N/A'):.1%}")
    print(f"  finish_reason_dist:        {stats.get('finish_reason_dist', {})}")
    print(f"  output_tokens p50/p90/p99: "
          f"{stats.get('output_tokens_p50','?')} / "
          f"{stats.get('output_tokens_p90','?')} / "
          f"{stats.get('output_tokens_p99','?')}")
    print(f"  gen_time p50/p90:          "
          f"{stats.get('generation_time_p50','?'):.3f}s / "
          f"{stats.get('generation_time_p90','?'):.3f}s")


=== Track A Inference Diagnostics ===
  truncation_rate:           40.0%
  empty_prediction_rate:     0.0%
  finish_reason_dist:        {'stop': 3, 'length': 2}
  output_tokens p50/p90/p99: 191.0 / 400.0 / 400.0
  gen_time p50/p90:          12.247s / 21.619s

=== Track B Inference Diagnostics ===
  truncation_rate:           20.0%
  empty_prediction_rate:     0.0%
  finish_reason_dist:        {'stop': 4, 'length': 1}
  output_tokens p50/p90/p99: 98.0 / 326.0 / 392.59999999999997
  gen_time p50/p90:          5.363s / 17.657s


## 8. LLM Judge

Sends each (question, gold answer, model answer) triple to a free LLM judge API
(Cerebras → Groq → Gemini, in fallback order). Returns per-axis scores (1–5),
hallucination classification, and PASS/FAIL/UNSAFE verdict.

> **Skipped automatically** if no judge API key is set.

In [15]:
has_judge_key = any([
    os.environ.get('CEREBRAS_API_KEY'),
    os.environ.get('GROQ_API_KEY'),
    os.environ.get('GEMINI_API_KEY'),
])

if not has_judge_key:
    print('No judge API key found — skipping LLM judge.')
    print('Add CEREBRAS_API_KEY, GROQ_API_KEY, or GEMINI_API_KEY to run this section.')
else:
    limit = cfg['dataset']['num_test']
    print(f'Judging up to {limit} rows per track...')

    for track in ('trackA', 'trackB'):
        cmd = (
            f"python llm_judge.py"
            f" --predictions outputs/{track}/predictions.csv"
            f" --output      outputs/{track}/judged.csv"
            f" --limit       {limit}"
        )
        print(f'\n$ {cmd}')
        ret = os.system(cmd)
        if ret != 0:
            print(f'  WARNING: llm_judge.py failed for {track} (exit {ret})')

Judging up to 5 rows per track...

$ python llm_judge.py --predictions outputs/trackA/predictions.csv --output      outputs/trackA/judged.csv --limit       5
[judge] loaded 5 predictions from outputs/trackA/predictions.csv
[judge] running in single-judge fallback mode. Use --all_judges for report-quality consensus.
[judge] initializing providers:
  + cerebras ready
  + groq ready
  + gemini ready
    [cerebras] failed: Error code: 404 - {'message': 'Model llama-3.3-70b does not exist or you do not have access to it.', 'type': 'not_found_
    [cerebras] failed: Error code: 404 - {'message': 'Model llama-3.3-70b does not exist or you do not have access to it.', 'type': 'not_found_
    [cerebras] failed: Error code: 404 - {'message': 'Model llama-3.3-70b does not exist or you do not have access to it.', 'type': 'not_found_
    [cerebras] failed: Error code: 404 - {'message': 'Model llama-3.3-70b does not exist or you do not have access to it.', 'type': 'not_found_
    [cerebras] failed: E

## 9. Safety Audit Templates

Generates blank CSV templates for manual safety review.
Rows are split across three risk buckets (low / medium / high).

**After running this cell:**
1. Download `outputs/trackA/safety_audit.csv` and `outputs/trackB/safety_audit.csv`
2. Fill in `clinical_correctness`, `risk_severity`, `hallucination_type`, `safe_behavior`, `manual_remark`
3. Re-upload and re-run Section 10 to include audit data in the final report

In [16]:
from src.safety_rubric import build_blank_audit_rows, make_audit_csv

def pick_audit_rows(pred_path, judged_path):
    pred_df = pd.read_csv(pred_path)
    if len(pred_df) == 0:
        raise ValueError(f'predictions file is empty: {pred_path}')

    track_name = str(pred_df.iloc[0]['track_name'])

    try:
        judge_df = pd.read_csv(judged_path)
        df = pred_df.merge(
            judge_df[['sample_id', 'any_unsafe', 'majority_pass',
                       'n_major_errors', 'max_severity', 'n_errors']],
            on='sample_id', how='left',
        )
        has_judge = True
    except (FileNotFoundError, KeyError):
        df = pred_df.copy()
        has_judge = False

    if has_judge:
        mask_high   = (df.get('any_unsafe', False) == True) | \
                      (df.get('n_major_errors', 0) > 0) | \
                      (df.get('max_severity', 0) >= 4)
        mask_medium = (~mask_high) & (
                      (df.get('majority_pass', True) == False) |
                      (df.get('n_errors', 0) > 0) |
                      (df.get('truncated', False) == True)
        )
    else:
        mask_high   = df.get('truncated', pd.Series([False] * len(df))) == True
        mask_medium = pd.Series([False] * len(df))

    mask_low = ~mask_high & ~mask_medium

    high_rows   = df[mask_high]
    medium_rows = df[mask_medium]
    low_rows    = df[mask_low].sample(frac=1, random_state=42)

    # For very small test sets (dry-run), ensure at least 1 row per bucket
    if len(high_rows) == 0 and len(df) > 0:
        high_rows   = df.iloc[[0]]
        medium_rows = df.iloc[1:2] if len(df) > 1 else medium_rows
        low_rows    = df.iloc[2:]  if len(df) > 2 else low_rows

    rows = []
    for bucket, part in [('high', high_rows), ('medium', medium_rows), ('low', low_rows)]:
        if part.empty:
            continue
        rows.extend(build_blank_audit_rows(
            part.to_dict('records'),
            track_name=track_name,
            risk_bucket=bucket,
        ))

    if not has_judge:
        print(f'  [{track_name}] judge output not found — used truncation as risk signal')
        print(f'    Re-run this section after judge completes for better bucket assignment.')

    print(f'  [{track_name}] high={len(high_rows)}  medium={len(medium_rows)}  low={len(low_rows)} rows selected')
    return rows

for track, pred, judged in [
    ('trackA', 'outputs/trackA/predictions.csv', 'outputs/trackA/judged.csv'),
    ('trackB', 'outputs/trackB/predictions.csv', 'outputs/trackB/judged.csv'),
]:
    make_audit_csv(
        pick_audit_rows(pred, judged),
        f'outputs/{track}/safety_audit.csv',
    )

print('\nAudit templates written:')
print('  outputs/trackA/safety_audit.csv')
print('  outputs/trackB/safety_audit.csv')

  [A] high=4  medium=1  low=0 rows selected
  [B] high=4  medium=1  low=0 rows selected

Audit templates written:
  outputs/trackA/safety_audit.csv
  outputs/trackB/safety_audit.csv


## 10. Final Report

Merges automatic metrics, judge scores, and audit results into the final comparison tables.
Judge and audit files are **optional** — the report runs even if they are missing.

In [17]:
def _load_csv(path):
    p = Path(path)
    if p.exists() and p.stat().st_size > 0:
        return pd.read_csv(p)
    print(f'  [missing] {path}')
    return pd.DataFrame()

judge_a = _load_csv('outputs/trackA/judged.csv')
judge_b = _load_csv('outputs/trackB/judged.csv')
audit_a = _load_csv('outputs/trackA/safety_audit.csv')
audit_b = _load_csv('outputs/trackB/safety_audit.csv')

print(f'pred_a:  {len(pred_a)} rows   pred_b:  {len(pred_b)} rows')
print(f'judge_a: {len(judge_a)} rows  judge_b: {len(judge_b)} rows')
print(f'audit_a: {len(audit_a)} rows  audit_b: {len(audit_b)} rows')

pred_a:  5 rows   pred_b:  5 rows
judge_a: 5 rows  judge_b: 5 rows
audit_a: 5 rows  audit_b: 5 rows


In [18]:
# Core comparison table
rows = []
for track, data in metrics_summary.items():
    rows.append({
        'track':                  track,
        'exact_match':            data.get('exact_match'),
        'rouge_l':                data.get('rouge_l'),
        'mean_output_tokens':     data.get('mean_output_tokens'),
        'mean_generation_time_s': data.get('mean_generation_time_s'),
        'mean_tokens_per_sec':    data.get('mean_tokens_per_sec'),
        'n':                      data.get('n'),
    })

comparison = pd.DataFrame(rows).set_index('track')
comparison.to_csv('outputs/final_comparison_table.csv')

print('=== Core Comparison ===')
comparison

=== Core Comparison ===


,exact_match,rouge_l,mean_output_tokens,mean_generation_time_s,mean_tokens_per_sec,n
track,,,,,,
trackA,0,0.081639,216.2,12.103,17.6,5
trackB,0,0.084473,160.0,8.736,18.0,5


In [19]:
# Per-sample merged table: Track A vs B side-by-side with deltas
from src.data_formatting import extract_answer_for_scoring
from src.metrics import compute_core_metrics

def _per_sample_metrics(df, track):
    preds = [extract_answer_for_scoring(p, track) for p in df['prediction']]
    refs  = list(df['reference'])
    em_list = [
        1.0 if p.strip().lower() == r.strip().lower() else 0.0
        for p, r in zip(preds, refs)
    ]
    return em_list

pa = pd.read_csv('outputs/trackA/predictions.csv')
pb = pd.read_csv('outputs/trackB/predictions.csv')

pa['em'] = _per_sample_metrics(pa, 'A')
pb['em'] = _per_sample_metrics(pb, 'B')

merged = pa[['sample_id', 'question', 'reference',
             'prediction', 'em', 'output_tokens', 'generation_time_s']].merge(
    pb[['sample_id', 'prediction', 'em', 'output_tokens', 'generation_time_s']],
    on='sample_id', suffixes=('_A', '_B'),
)
merged['em_delta_A_minus_B']      = merged['em_A'] - merged['em_B']
merged['token_delta_A_minus_B']   = merged['output_tokens_A'] - merged['output_tokens_B']
merged['latency_delta_A_minus_B'] = merged['generation_time_s_A'] - merged['generation_time_s_B']

merged.to_csv('outputs/per_sample_comparison.csv', index=False)
print(f'Per-sample table: {len(merged)} rows -> outputs/per_sample_comparison.csv')

# Case breakdown
a_better  = int((merged['em_delta_A_minus_B'] >  0).sum())
b_better  = int((merged['em_delta_A_minus_B'] <  0).sum())
tied      = int((merged['em_delta_A_minus_B'] == 0).sum())
trunc_a   = int(pa.get('truncated', pd.Series([False]*len(pa))).sum())
trunc_b   = int(pb.get('truncated', pd.Series([False]*len(pb))).sum())

print(f'\n=== Case Breakdown ===')
print(f'  A better  (EM A=1, B=0): {a_better}')
print(f'  B better  (EM B=1, A=0): {b_better}')
print(f'  Tied:                    {tied}')
print(f'  Truncated A / B:         {trunc_a} / {trunc_b}')

Per-sample table: 5 rows -> outputs/per_sample_comparison.csv

=== Case Breakdown ===
  A better  (EM A=1, B=0): 0
  B better  (EM B=1, A=0): 0
  Tied:                    5
  Truncated A / B:         2 / 1


In [ ]:
# Worked examples: 3 A-wins, 3 B-wins; fall back to tied/all when not enough winners
def _show_example(row, idx, label=''):
    print(f'\n--- Example {idx}{(" — " + label) if label else ""} ---')
    print(f'Q:         {str(row["question"])[:300]}')
    print(f'Reference: {str(row["reference"])[:200]}')
    print(f'Track A:   {str(row["prediction_A"])[:300]}')
    print(f'Track B:   {str(row["prediction_B"])[:300]}')
    print(f'EM A={row["em_A"]:.0f}  EM B={row["em_B"]:.0f}  '
          f'tokens A={row["output_tokens_A"]}  B={row["output_tokens_B"]}  '
          f'trunc A={row.get("truncated_A", "?")}  B={row.get("truncated_B", "?")}')

a_wins = merged[merged['em_delta_A_minus_B'] >  0]
b_wins = merged[merged['em_delta_A_minus_B'] <  0]
tied   = merged[merged['em_delta_A_minus_B'] == 0]

if not a_wins.empty:
    print('===== Track A (CoT) wins =====')
    for i, (_, row) in enumerate(a_wins.head(3).iterrows(), 1):
        _show_example(row, i, 'A better')
else:
    print('(no A-wins in this test set)')

if not b_wins.empty:
    print('\n===== Track B (no CoT) wins =====')
    for i, (_, row) in enumerate(b_wins.head(3).iterrows(), 1):
        _show_example(row, i, 'B better')
else:
    print('(no B-wins in this test set)')

# Always show up to 3 tied examples for qualitative inspection
if not tied.empty:
    print('\n===== Tied examples (qualitative inspection) =====')
    for i, (_, row) in enumerate(tied.head(3).iterrows(), 1):
        _show_example(row, i, 'tied')

In [21]:
COMPARABLE_AXES = ['mean_clinical_correctness', 'mean_factuality',
                   'mean_completeness', 'mean_safety']

def judge_summary(df):
    if df.empty:
        return {'note': 'no judge results yet'}
    out = {}
    # Comparable axes: always present for both tracks
    for col in COMPARABLE_AXES:
        if col in df.columns and df[col].notna().any():
            out[col] = round(float(df[col].mean()), 3)
    # reasoning_soundness: Track A only — exclude nulls from mean
    rs_col = 'mean_reasoning_soundness'
    if rs_col in df.columns and df[rs_col].notna().any():
        n_valid = df[rs_col].notna().sum()
        out[rs_col] = round(float(df[rs_col].mean()), 3)
        out[f'{rs_col}_n'] = int(n_valid)
    # Error + verdict aggregates
    for col in ['n_major_errors', 'max_severity', 'n_errors']:
        if col in df.columns:
            out[col] = round(float(df[col].mean()), 3)
    if 'any_unsafe' in df.columns:
        out['unsafe_rate']    = round(float(df['any_unsafe'].mean()), 3)
    if 'majority_pass' in df.columns:
        out['pass_rate']      = round(float(df['majority_pass'].mean()), 3)
    return out

def audit_summary(df):
    if df.empty:
        return {'note': 'no audit results yet'}
    return {
        'risk_severity':      df['risk_severity'].value_counts(dropna=False).to_dict()      if 'risk_severity'      in df.columns else {},
        'hallucination_type': df['hallucination_type'].value_counts(dropna=False).to_dict() if 'hallucination_type' in df.columns else {},
        'safe_behavior':      df['safe_behavior'].value_counts(dropna=False).to_dict()      if 'safe_behavior'      in df.columns else {},
    }

report_summary = {
    'judge_trackA': judge_summary(judge_a),
    'judge_trackB': judge_summary(judge_b),
    'audit_trackA': audit_summary(audit_a),
    'audit_trackB': audit_summary(audit_b),
}

Path('outputs/report_summary.json').write_text(
    json.dumps(report_summary, indent=2), encoding='utf-8'
)
print(json.dumps(report_summary, indent=2))

{
  "judge_trackA": {
    "mean_clinical_correctness": 1.8,
    "mean_factuality": 1.8,
    "mean_completeness": 1.2,
    "mean_safety": 3.4,
    "n_major_errors": 1.8,
    "max_severity": 4.6,
    "n_errors": 2.2,
    "unsafe_rate": 0.0,
    "pass_rate": 0.0
  },
  "judge_trackB": {
    "mean_clinical_correctness": 2.2,
    "mean_factuality": 2.6,
    "mean_completeness": 1.2,
    "mean_safety": 3.4,
    "n_major_errors": 1.2,
    "max_severity": 3.8,
    "n_errors": 1.2,
    "unsafe_rate": 0.0,
    "pass_rate": 0.0
  },
  "audit_trackA": {
    "risk_severity": {
      "NaN": 5
    },
    "hallucination_type": {
      "NaN": 5
    },
    "safe_behavior": {
      "NaN": 5
    }
  },
  "audit_trackB": {
    "risk_severity": {
      "NaN": 5
    },
    "hallucination_type": {
      "NaN": 5
    },
    "safe_behavior": {
      "NaN": 5
    }
  }
}


In [ ]:
# ── Phase 3 Deliverable: Error Analysis Document ──────────────────────────
print('=' * 60)
print('  ERROR ANALYSIS — Phase 3 Deliverable')
print('=' * 60)

_HAL_TYPES = ['fabrication', 'negation', 'causality', 'contextual', 'reasoning']

for track_name, jdf in [('Track A (CoT)', judge_a), ('Track B (no CoT)', judge_b)]:
    print(f'\n── {track_name} ──')
    if jdf.empty:
        print('  No judge data.')
        continue

    # Failure mode breakdown
    total = len(jdf)
    n_fail   = int((jdf.get('majority_pass', pd.Series([True]*total)) == False).sum())
    n_unsafe = int(jdf.get('any_unsafe', pd.Series([False]*total)).sum())
    n_trunc  = int(pd.read_csv(
        'outputs/trackA/predictions.csv' if 'A' in track_name else 'outputs/trackB/predictions.csv'
    ).get('truncated', pd.Series([False]*total)).sum())

    print(f'  Total samples:       {total}')
    print(f'  FAIL verdicts:       {n_fail} / {total}  ({n_fail/total:.0%})')
    print(f'  UNSAFE verdicts:     {n_unsafe} / {total}  ({n_unsafe/total:.0%})')
    print(f'  Truncated outputs:   {n_trunc} / {total}  ({n_trunc/total:.0%})')

    # Hallucination type breakdown
    print(f'\n  Hallucination types:')
    for ht in _HAL_TYPES:
        col = f'n_{ht}'
        if col in jdf.columns:
            cnt = int(jdf[col].sum())
            print(f'    {ht:12s}  {cnt:3d}')

    # Score summary
    print(f'\n  Judge scores (mean / 5):')
    for ax in ['mean_clinical_correctness', 'mean_factuality',
               'mean_completeness', 'mean_safety']:
        if ax in jdf.columns and jdf[ax].notna().any():
            print(f'    {ax:30s}  {jdf[ax].mean():.2f}')

    # Where the model fails most
    worst = jdf.nlargest(min(2, len(jdf)), 'n_major_errors')
    if not worst.empty and 'n_major_errors' in worst.columns:
        print(f'\n  Samples with most major errors:')
        for _, r in worst.iterrows():
            sid = int(r.get('sample_id', -1))
            nm  = int(r.get('n_major_errors', 0))
            ms  = float(r.get('max_severity', 0))
            print(f'    sample_id={sid}  major_errors={nm}  max_severity={ms:.0f}')

print('\nSee outputs/*/judged.csv for full per-sample detail.')
print('Fill outputs/*/safety_audit.csv manually for the non-clinical safety audit.')

## Done — Outputs

| File | Contents |
|------|----------|
| `outputs/trackA/predictions.csv` | Track A predictions + latency |
| `outputs/trackB/predictions.csv` | Track B predictions + latency |
| `outputs/trackA/judged.csv` | LLM judge scores (if run) |
| `outputs/trackB/judged.csv` | LLM judge scores (if run) |
| `outputs/trackA/safety_audit.csv` | Blank audit template (fill manually) |
| `outputs/trackB/safety_audit.csv` | Blank audit template (fill manually) |
| `outputs/metrics_summary.json` | EM + ROUGE-L per track |
| `outputs/final_comparison_table.csv` | Side-by-side comparison |
| `outputs/report_summary.json` | Judge + audit summaries |

---

**Assignment questions to answer from these outputs:**
- *Does reasoning improve QA?* → compare `exact_match` and `rouge_l`
- *Does reasoning increase cost/latency?* → compare `mean_output_tokens` and `mean_generation_time_s`
- *Does reasoning increase hallucinations?* → compare judge error counts and `hallucination_type`
- *Should reasoning be hidden or shown?* → discuss `reasoning_clarity` × `safe_behavior` from audit

> *This is a learning artefact. Not a clinical product.*

## 11. Save Outputs

In [ ]:
import shutil, datetime, glob

# Zip the entire outputs/ directory so it appears in the Kaggle Output tab for download.
_ts       = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
_zip_base = f'/kaggle/working/pipeline_outputs_{_ts}'
_src_dir  = 'outputs'

print('Zipping outputs/ ...')
shutil.make_archive(_zip_base, 'zip', root_dir='.', base_dir=_src_dir)
_zip_file = _zip_base + '.zip'
_size_mb  = Path(_zip_file).stat().st_size / 1_000_000

print(f'\n✅  {_zip_file}  ({_size_mb:.1f} MB)')
print('   ↳ Download from the Kaggle notebook Output tab (right sidebar → Output → Files)')

# Also list each file that was included
print('\nFiles zipped:')
for _f in sorted(glob.glob('outputs/**/*', recursive=True)):
    if Path(_f).is_file():
        _kb = Path(_f).stat().st_size / 1000
        print(f'  {_f}  ({_kb:.1f} kB)')